In [19]:
import pandas as pd

In [20]:
file="/Users/shivaya/Desktop/Research/DelhiMetroResearch/DataRaw/dmrc_5months.xlsx"
excel = pd.ExcelFile(file)

In [21]:
sep_entry = pd.read_excel(
    file, 
    sheet_name="september_2024_entry")

In [22]:
print(sep_entry.columns.tolist())

['businessday', 'linename', 'sitename', 'station name', 'station code', 'HR4', 'HR5', 'HR6', 'HR7', 'HR8', 'HR9', 'HR10', 'HR11', 'HR12', 'HR13', 'HR14', 'HR15', 'HR16', 'HR17', 'HR18', 'HR19', 'HR20', 'HR21', 'HR22', 'HR23', 'HR0']


In [23]:
sep_long = sep_entry.melt(
    id_vars=[
        "businessday",
        "linename",
        "sitename",
        "station name",
        "station code"
    ],
    var_name="hour",
    value_name="entries"
)

In [24]:
sep_long.head()

,businessday,linename,sitename,station name,station code,hour,entries
0,2024-09-01,RMGL,Sector 55-56-RMGL,Sector 55-56,Sector 55-56,HR4,0
1,2024-09-01,LINE06,Nehru Place,Nehru Place,NP,HR4,0
2,2024-09-01,LINE06,Kalkaji Mandir,Kalkaji Mandir,KJMD,HR4,0
3,2024-09-01,LINE06,Gobind Puri,Govindpuri,GDPI,HR4,0
4,2024-09-01,LINE06,Okhla,Harkesh Nagar Okhla,OK,HR4,0


In [25]:
excel.sheet_names
sheet_names = excel.sheet_names[1:]

In [26]:
all_data= []

In [27]:
all_data = []

for sheet in sheet_names:

    # Read worksheet
    df = pd.read_excel(file, sheet_name=sheet)

    # Convert wide format to long format
    df = df.melt(
        id_vars=[
            "businessday",
            "linename",
            "sitename",
            "station name",
            "station code"
        ],
        var_name="hour",
        value_name="count"
    )

    # Convert HR4 -> 4
    df["hour"] = (
        df["hour"]
        .str.replace("HR", "", regex=False)
        .astype(int)
    )

    # Add metadata
    df["month"] = sheet.split("_")[0].capitalize()
    df["direction"] = sheet.split("_")[-1].capitalize()

    all_data.append(df)

In [28]:
sheet_names

['september_2024_entry',
 'september_2024_exit',
 'october_2024_entry',
 'october_2024_exit',
 'november_2024_entry',
 'november_2024_exit',
 'december_2024_entry',
 'december_2024_exit',
 'january_2025_entry',
 'january_2025_exit',
 'february_2025_entry',
 'february_2025_exit']

In [29]:
metro_df = pd.concat(all_data, ignore_index=True)

In [30]:
metro_df.groupby("station name")

In [31]:
metro_df.groupby("station name")["count"].sum()

station name
Adarsh Nagar         3585769
Aiims                8674730
Akshardham           5852849
Anand Vihar Isbt    13246757
Arjan Garh           3786792
                      ...   
Vidhan Sabha         2401388
Vinobapuri           3146004
Vishwavidyalaya      9377227
Welcome              2881248
Yamuna Bank           191366
Name: count, Length: 237, dtype: int64

In [32]:
metro_df.groupby("station name")["count"].mean()

station name
Adarsh Nagar         471.687582
Aiims               1141.111550
Akshardham           769.909103
Anand Vihar Isbt    1742.535780
Arjan Garh           498.131018
                       ...     
Vidhan Sabha         315.888977
Vinobapuri           413.838990
Vishwavidyalaya     1233.521047
Welcome              379.011839
Yamuna Bank           25.173112
Name: count, Length: 237, dtype: float64

In [33]:
metro_df.groupby("station name")["count"].sum().sort_values(ascending=False)

station name
New Delhi                          24603150
Chandni Chowk                      24518011
Millennium City Centre Gurugram    20162410
Kashmere Gate                      19960586
Rajiv Chowk                        19029404
                                     ...   
Phase 2                              453318
Moulsari Avenue                      405505
Phase 1                              404127
Yamuna Bank                          191366
Krishna Park Extension               154782
Name: count, Length: 237, dtype: int64

In [34]:
metro_df.groupby("hour")["count"].sum().sort_values(ascending=False)

hour
18    101452774
9     100357290
19     95808450
10     82002968
17     78841046
8      67530915
20     67119871
16     64094654
11     62310943
14     62306593
15     61786557
13     61040022
12     59973792
21     39037598
7      33470095
22     19979674
6      11807315
23      6913333
5       2275883
0        707790
4         24245
Name: count, dtype: int64

In [35]:
metro_df.groupby("linename")["count"].sum().sort_values(ascending=False)

linename
LINE02    295077895
LINE03    250549361
LINE06    114033910
LINE01    111685951
LINE07    100136023
LINE08     84336061
LINE04     54830144
LINE05     51269905
RMGL        9719792
LINE09      7202766
Name: count, dtype: int64

In [36]:
metro_df.groupby("month")["count"].sum().sort_values(ascending=False)

month
September    186243377
January      185854056
October      185708886
December     180885924
November     176077149
February     164072416
Name: count, dtype: int64

In [37]:
## which station has the highest entries in a month
station_avg = (
    metro_df.groupby("station name")["count"]
    .mean()
    .sort_values(ascending=False)
)
station_avg.head(10)

station name
New Delhi                          3236.404893
Chandni Chowk                      3225.205341
Millennium City Centre Gurugram    2652.250723
Rajiv Chowk                        2503.210208
Karol Bagh                         2078.101421
Saket                              1831.466062
Dwarka Mor                         1791.889108
Anand Vihar Isbt                   1742.535780
Sector - 52 Noida                  1733.107077
Guru Teg Bahadur Nagar             1722.374901
Name: count, dtype: float64

In [38]:
## which is the busiest hour of at rajiv chowk station
rajiv = metro_df[metro_df["station name"] == "Rajiv Chowk"]
rajiv.head()

,businessday,linename,sitename,station name,station code,hour,count,month,direction
137,2024-09-01,LINE02,Rajiv Chowk,Rajiv Chowk,RCK,4,0,September,Entry
380,2024-09-02,LINE02,Rajiv Chowk,Rajiv Chowk,RCK,4,0,September,Entry
628,2024-09-03,LINE02,Rajiv Chowk,Rajiv Chowk,RCK,4,0,September,Entry
875,2024-09-04,LINE02,Rajiv Chowk,Rajiv Chowk,RCK,4,0,September,Entry
1120,2024-09-05,LINE02,Rajiv Chowk,Rajiv Chowk,RCK,4,0,September,Entry


In [39]:
rajiv.groupby("hour")["count"].mean().sort_values(ascending=False)

hour
18    5668.091160
17    5136.522099
19    4627.549724
16    4495.889503
15    4147.055249
14    3923.845304
13    3608.511050
20    3317.417127
10    3168.662983
12    3095.447514
11    2739.925414
9     2713.917127
21    2285.251381
22    1228.389503
8     1122.008287
7      579.290055
23     436.842541
6      212.055249
5       50.516575
0        9.875691
4        0.350829
Name: count, dtype: float64

In [40]:
#are entries higher than exits
metro_df.groupby("direction")["count"].mean()

direction
Entry    554.180000
Exit     598.143028
Name: count, dtype: float64

In [41]:
## busiest month of these 5 months is 
metro_df.groupby("month")["count"].mean().sort_values(ascending=False)

month
September    600.862618
October      579.811191
January      578.217244
November     568.064102
February     564.847614
December     564.642113
Name: count, dtype: float64

In [42]:
#which is the busiest hour at rajiv chowk station
rajiv.groupby("hour")["count"].mean().sort_values(ascending=False)

hour
18    5668.091160
17    5136.522099
19    4627.549724
16    4495.889503
15    4147.055249
14    3923.845304
13    3608.511050
20    3317.417127
10    3168.662983
12    3095.447514
11    2739.925414
9     2713.917127
21    2285.251381
22    1228.389503
8     1122.008287
7      579.290055
23     436.842541
6      212.055249
5       50.516575
0        9.875691
4        0.350829
Name: count, dtype: float64

In [43]:
# Which station has the highest entries in September?
metro_df[
    (metro_df["month"] == "September") &
    (metro_df["direction"] == "Entry")
].groupby("station name")["count"].sum().sort_values(ascending=False)

station name
New Delhi                          1914942
Millennium City Centre Gurugram    1813055
Chandni Chowk                      1705521
Kashmere Gate                      1621421
Rajiv Chowk                        1506303
                                    ...   
Shankar Vihar                        37321
Phase 1                              36983
Belvedere Towers                     36102
Moulsari Avenue                      35367
Yamuna Bank                          16394
Name: count, Length: 236, dtype: int64

In [44]:
# Which station has the highest exits in September?
metro_df[
    (metro_df["month"] == "September") &
    (metro_df["direction"] == "Exit")
].groupby("station name")["count"].sum().sort_values(ascending=False)



station name
New Delhi                          2160304
Chandni Chowk                      2049884
Millennium City Centre Gurugram    1717128
Kashmere Gate                      1647792
Rajiv Chowk                        1614239
                                    ...   
Punjabi Bagh                         48700
Moulsari Avenue                      39398
Shankar Vihar                        38829
Phase 1                              37808
Yamuna Bank                          16327
Name: count, Length: 236, dtype: int64

In [46]:
metro_df.to_csv("/Users/shivaya/Desktop/Research/DelhiMetroResearch/DataProcessed/metro_master.csv", index=False)